## Cleaning 02_nav_history.csv

In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/02_nav_history.csv")
print(df['date'].dtype)

str


In [4]:
# convert date to datetime
df['date'] = pd.to_datetime(df['date'])


In [5]:
# Sort by amfi_code + date
df = df.sort_values(["amfi_code", "date"])

In [6]:
# Forward fill NAV

df['nav'] = df.groupby('amfi_code')['nav'].ffill()


In [7]:
# Remove duplicates
df = df.drop_duplicates()

In [8]:
# Keep only valid NAV values
df = df[df['nav'] > 0]

In [9]:
# Save cleaned file
df.to_csv(
    "../data/processed/02_nav_history_clean.csv",
    index=False
)


## Cleaning 08_investor_transactions.csv

In [10]:
import pandas as pd
df_tx = pd.read_csv("../data/raw/08_investor_transactions.csv")
df_tx.head()
df_tx.info()
df_tx['transaction_date'].dtype


<class 'pandas.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  str    
 1   transaction_date    32778 non-null  str    
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  str    
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  str    
 6   city                32778 non-null  str    
 7   city_tier           32778 non-null  str    
 8   age_group           32778 non-null  str    
 9   gender              32778 non-null  str    
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  str    
 12  kyc_status          32778 non-null  str    
dtypes: float64(1), int64(2), str(10)
memory usage: 5.4 MB


<StringDtype(na_value=nan)>

In [11]:
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])
df_tx['transaction_date'].dtype


dtype('<M8[us]')

In [ ]:
# Standardise Transaction Types

df_tx['transaction_type'] = (
    df_tx['transaction_type']
    .astype(str)
    .str.strip()
    .str.lower()
)

# Map SIP, Lumpsum, Redemption properly
type_map = { 'sip':'SIP','lumpsum':'Lumpsum','redemption':'Redemption'}

df_tx['transaction_type'] = df_tx['transaction_type'].map(type_map).fillna(df_tx['transaction_type'])
#Verify
df_tx['transaction_type'].unique()


<ArrowStringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str

In [13]:
#  Validate amount_inr > 0
df_tx = df_tx[df_tx["amount_inr"] > 0]


In [14]:
# Validate KYC Status

df_tx['kyc_status'] = (df_tx['kyc_status'].astype(str).str.strip().str.title())

valid_kyc = [ 
    'Verified',
    'Pending',
    'Failed']

df_tx = df_tx[df_tx['kyc_status'].isin(valid_kyc)]
df_tx['kyc_status'].unique()




<ArrowStringArray>
['Verified', 'Pending']
Length: 2, dtype: str

In [ ]:
#  Fix date formats
df_tx["transaction_date"] = pd.to_datetime(df_tx["transaction_date"])
df_tx["transaction_date"] = df_tx["transaction_date"].dt.strftime("%Y-%m-%d")

In [16]:
# Remove Duplicates
df.duplicated().sum()
df_tx = df_tx.drop_duplicates()
df_tx.isnull().sum()
df_tx = df_tx.dropna()

In [ ]:
# Save Cleaned File
df_tx.to_csv( "../data/processed/08_investor_transactions_clean.csv",index=False)